## Building a RAG System
### 2.1 System Architecture
#### 2.1.1 Core Modules
Indexing: document loading -> text chunking -> embedding -> vector storage.

Retrieval: query embedding -> similarity search -> context extraction.

Generation: prompt assembly -> LLM inference -> response post-processing.

Service layer: REST endpoint -> streaming response -> runtime monitoring.

#### 2.1.2 How I built this RAG system (implementation summary)
1. Prepared local academic papers under `knowledge_base/` (PDF-only).
2. Parsed PDFs with `UnstructuredPDFLoader` and normalized content/metadata.
3. Split text into overlapping chunks (`chunk_size=500`, `chunk_overlap=50`).
4. Generated embeddings using `bge-base-zh-v1.5`.
5. Built a persistent Chroma vector store in `vectorstore/`.
6. Configured retrieval with similarity search and top-k candidates.
7. Added a query-rewriting step to improve follow-up question retrieval.
8. Connected retriever + prompt + LLM (`qwen-turbo`) as a complete RAG chain.
9. Exposed the pipeline with FastAPI and a simple streaming web demo.
10. Validated the workflow using domain-specific QA prompts and screenshots in `figs/`.

#### 2.1.2 Core Stack

| Component | Selection | Notes |
|---|---|---|
| Embedding model | SentenceTransformers | Sentence-level dense vectors for semantic similarity. |
| Vector database | Chroma | Lightweight local vector store with persistence and metadata. |
| Alternatives | FAISS / Milvus | Optional backends for different scale requirements. |
| LLM runtime | Tongyi API (`qwen-turbo`) | API-based generation backend in this project. |
| Service framework | FastAPI | Async endpoints with streaming response support. |


#### 2.1.3 Environment Setup

In [1]:
# Use system Python + pip (conda is optional).
# If multiple Python versions exist, confirm Python 3.10+ first.

In [2]:
# Install dependencies (example):
# pip install -r requirements.txt

### 2.2 Indexing Pipeline
#### 2.2.1 Document Loading (PDF-only)
For this project submission, we only use local PDF files as the data source.
The notebook keeps a single PDF loading path to align with the course task (academic-paper RAG).

Loader used in this notebook:
- UnstructuredPDFLoader (PDF parsing)

Other loaders (txt / markdown / docx / web) are removed from the final workflow.

1) Load PDF documents

In [3]:
"""
Load one PDF file as LangChain Documents.
Each Document contains:
- metadata: source and parsing metadata
- page_content: extracted text
"""

from langchain_community.document_loaders import UnstructuredPDFLoader

pdf_documents = UnstructuredPDFLoader(
    "knowledge_base/Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf",
    mode="elements",
    strategy="hi_res",
    languages=["eng"],
).load()
print(pdf_documents[:3])

d:\PY3\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': 'knowledge_base/Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf', 'detection_class_prob': 0.888597309589386, 'coordinates': {'points': ((np.float64(135.68545532226562), np.float64(83.45243072509766)), (np.float64(135.68545532226562), np.float64(102.07794952392578)), (np.float64(775.6200561523438), np.float64(102.07794952392578)), (np.float64(775.6200561523438), np.float64(83.45243072509766))), 'system': 'PixelSpace', 'layout_width': 1700, 'layout_height': 2200}, 'last_modified': '2026-03-17T15:16:41', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'file_directory': 'knowledge_base', 'filename': 'Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf', 'category': 'Header', 'element_id': 'da447b0189729dde746991b22050e7e3'}, page_content='IEEE TRANSACTIONS ON CYBERNETICS, VOL. 55, NO. 3, MARCH 2025'), Document(metadata={'source': 'knowledge_base/Dynamic_Graph_Representat

2) PDF loading options

(Option A) Parse a PDF with `PyPDFLoader` (quick baseline).

In [4]:
"""
PyPDFLoader baseline example.
- Splits a PDF by page into LangChain Documents.
- `extraction_mode="layout"` keeps coarse layout information.
"""

from langchain_community.document_loaders import PyPDFLoader

pdf_documents = PyPDFLoader(
    "knowledge_base/Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf",
    extraction_mode="layout",
).load()
print(pdf_documents[:3])

[Document(metadata={'producer': 'Acrobat Distiller 23.0 (Windows); modified using iText® Core 7.2.4 (AGPL version) ©2000-2022 iText Group NV', 'creator': 'LaTeX with hyperref package', 'creationdate': '2025-02-26T17:49:52+05:30', 'moddate': '2025-03-05T21:23:30-05:00', 'ieee article id': '10870416', 'ieee issue id': '10916543', 'subject': 'IEEE Transactions on Cybernetics;2025;55;3;10.1109/TCYB.2025.3531657', 'ieee publication id': '6221036', 'title': 'Dynamic Graph Representation Learning for Spatio-Temporal Neuroimaging Analysis', 'source': 'knowledge_base/Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1121'}, page_content='IEEE         TRANSACTIONS         ON         CYBERNETICS,         VOL.         55,         NO.         3,         MARCH         2025                                                                                                                                                      

(Option B) Parse PDF files with `UnstructuredPDFLoader`

If you use `UnstructuredPDFLoader` with `hi_res`, install Poppler and Tesseract OCR first.

- Poppler: PDF rendering/parsing backend.
- Tesseract OCR: extracts text from image-heavy pages.

After installation, add executable paths to your system `PATH`.

Example: parse PDF with `UnstructuredPDFLoader`:

In [5]:
"""
UnstructuredPDFLoader example.
- Supports structured extraction and OCR-ready parsing.
- `mode="elements"` parses title/list/table-like elements.
- `strategy="hi_res"` performs layout-aware extraction.
- `infer_table_structure=True` keeps table-related structure metadata.
- `languages` specifies OCR language codes.
More details: https://github.com/Unstructured-IO/unstructured/blob/main/unstructured/partition/pdf.py
"""

from langchain_community.document_loaders import UnstructuredPDFLoader

pdf_documents = UnstructuredPDFLoader(
    "knowledge_base/Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf",
    mode="elements",
    strategy="hi_res",
    infer_table_structure=True,
    languages=["eng"],
).load()
print(pdf_documents[:3])

The `max_size` parameter is deprecated and will be removed in v4.26. Please specify in `size['longest_edge'] instead`.


[Document(metadata={'source': 'knowledge_base/Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf', 'detection_class_prob': 0.888597309589386, 'coordinates': {'points': ((np.float64(135.68545532226562), np.float64(83.45243072509766)), (np.float64(135.68545532226562), np.float64(102.07794952392578)), (np.float64(775.6200561523438), np.float64(102.07794952392578)), (np.float64(775.6200561523438), np.float64(83.45243072509766))), 'system': 'PixelSpace', 'layout_width': 1700, 'layout_height': 2200}, 'last_modified': '2026-03-17T15:16:41', 'filetype': 'application/pdf', 'languages': ['eng'], 'page_number': 1, 'file_directory': 'knowledge_base', 'filename': 'Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf', 'category': 'Header', 'element_id': 'da447b0189729dde746991b22050e7e3'}, page_content='IEEE TRANSACTIONS ON CYBERNETICS, VOL. 55, NO. 3, MARCH 2025'), Document(metadata={'source': 'knowledge_base/Dynamic_Graph_Representat

When `strategy="hi_res"` is enabled, layout models may be downloaded automatically and cached by Hugging Face.

Typical cache path on Windows:
`C:\\Users\\<your_user_name>\\.cache\\huggingface\\hub\\`

2) Build a reusable PDF-only loading function

In [6]:
from pathlib import Path
from langchain_community.document_loaders import UnstructuredPDFLoader


def load_documents():
    """Load all PDF files from knowledge_base/ as LangChain Documents."""
    knowledge_dir = Path("knowledge_base")
    pdf_files = sorted([p for p in knowledge_dir.iterdir() if p.is_file() and p.suffix.lower() == ".pdf"])

    if not pdf_files:
        raise ValueError("No PDF files found in knowledge_base/.")

    documents = []
    for pdf_path in pdf_files:
        loaded = UnstructuredPDFLoader(
            str(pdf_path),
            mode="elements",
            strategy="hi_res",
            languages=["eng"],
        ).load()
        documents.extend(loaded)

    print(f"Loaded PDF files: {len(pdf_files)}, parsed document chunks: {len(documents)}")
    return documents


documents = load_documents()

Loaded PDF files: 4, parsed document chunks: 1004


7) Text Cleaning

Clean extracted text and normalize metadata before indexing.

Chroma metadata supports only primitive types (`str`, `int`, `float`, `bool`),
so unsupported metadata values are converted to JSON strings.

In [7]:
import re
import json


def clean_content(documents: list):
    """Clean text content and normalize metadata for Chroma compatibility."""
    cleaned_docs = []

    for doc in documents:
        # 1) Normalize text by collapsing repeated newlines and trimming spaces.
        text = doc.page_content
        text = re.sub(r"\n{2,}", "\n\n", text)
        text = text.strip()

        # 2) Convert unsupported metadata values to JSON strings.
        allowed_types = (str, int, float, bool)
        for key, value in doc.metadata.items():
            if not isinstance(value, allowed_types):
                try:
                    doc.metadata[key] = json.dumps(value, ensure_ascii=False, default=str)
                except (TypeError, ValueError):
                    doc.metadata[key] = str(value)

        # 3) Save normalized text back to the document.
        doc.page_content = text
        cleaned_docs.append(doc)

    return cleaned_docs

#### 2.2.2 Text Chunking
Loaded documents are usually too long for direct LLM context windows.
We split long text into smaller chunks before indexing.

`langchain-text-splitters` provides multiple splitting strategies.
In this notebook we use `RecursiveCharacterTextSplitter`.

Two key parameters:
- `chunk_size`: maximum length of each chunk
- `chunk_overlap`: overlap length between adjacent chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

cleaned_docs = clean_content(documents)

# Split text into chunks for retrieval.
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", ". "],
    chunk_size=500,
    chunk_overlap=50,
)
texts = text_splitter.split_documents(cleaned_docs)

#### 2.2.3 Embedding and Vector Storage
We use `bge-base-zh-v1.5` to convert text chunks into dense vectors.
These embeddings are stored in Chroma for semantic retrieval.

In [9]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

# Load embedding model.
embedding_model = HuggingFaceEmbeddings(
    model_name="./bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # Normalize vectors for cosine similarity.
)

# Extract text from Documents.
page_content_list = [text.page_content for text in texts]
# Embed all chunks.
embeddings = embedding_model.embed_documents(page_content_list)
# Preview embeddings.
for i, (page_content, vector) in enumerate(zip(page_content_list, embeddings)):
    print("Text:\n", page_content)
    print("Embedding:\n", vector[:5])
    print()
    if i == 5:
        break

Text:
 IEEE TRANSACTIONS ON CYBERNETICS, VOL. 55, NO. 3, MARCH 2025
Embedding:
 [-0.03756764531135559, -0.001977087464183569, -0.006297062151134014, 0.042036689817905426, -0.014929418452084064]

Text:
 Dynamic Graph Representation Learning for Spatio-Temporal Neuroimaging Analysis
Embedding:
 [-0.018714897334575653, -0.00616712961345911, 0.021035967394709587, 0.0010120182996615767, -0.001451089745387435]

Text:
 Rui Liu®, Member, IEEE, Yao Hu®, Member, IEEE, Jibin Wu®, Member, IEEE, Ka-Chun Wong®, Zhi-An Huang®, Member, IEEE, Yu-An Huang®, Member, IEEE, and Kay Chen Tan®, Fellow, IEEE
Embedding:
 [-0.017837781459093094, -0.014813967980444431, 0.009966474026441574, 0.01023274939507246, 0.0002163450262742117]

Text:
 Abstract—Neuroimaging analysis aims to reveal the information-processing mechanisms of the human brain in a noninvasive manner. In the past, graph neural networks (GNNs) have shown promise in capturing the non-Euclidean structure of brain networks. However, existing neuroima

Store embedded chunks in Chroma.

In Chroma, data is organized as a collection (similar to a table).
Each record typically contains:
- unique ID
- embedding vector
- source document chunk
- metadata key-value pairs

In [10]:
from langchain_chroma import Chroma

# Embed and persist documents in the vector database.
vectorstore = Chroma.from_documents(
    texts,  # Document chunks
    embedding_model,  # Embedding model
    persist_directory="vectorstore",  # Storage directory
)

Use `get()` to inspect stored data in Chroma.

In [11]:
print(vectorstore.get().keys())  # Show available fields
print(vectorstore.get(include=["embeddings"])["embeddings"][:5, :5])  # Preview embedding vectors

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])
[[-3.75676677e-02 -1.97711214e-03 -6.29708217e-03  4.20366786e-02
  -1.49294669e-02]
 [-1.87148973e-02 -6.16712961e-03  2.10359674e-02  1.01201830e-03
  -1.45108975e-03]
 [-1.98162403e-02 -1.30252866e-02  7.29821995e-03  1.08553562e-02
   4.93047468e-04]
 [ 2.66242251e-02  5.28688310e-03  4.48420867e-02  2.24639736e-02
  -1.74036843e-03]
 [ 1.11565255e-02 -7.31559849e-05  7.59461708e-03 -1.43060647e-02
  -3.01323039e-03]]


### 2.3 Retrieval
A user query is embedded into a vector and compared with vectors in the database (cosine similarity).
The system returns the most relevant document chunks.

First, load the embedding model and vector database.

In [12]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Load embedding model.
embedding_model = HuggingFaceEmbeddings(
    model_name="./bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Initialize Chroma client.
vectorstore = Chroma(
    persist_directory="vectorstore",
    embedding_function=embedding_model,
)

Retrieve documents that are similar to the user query.

In [13]:
# Similarity search
query = "How does the SpatialTemporal Co-Attention method model temporal and spatial information?"
sim_docs = vectorstore.similarity_search(query, k=3)  # Return top 3 results
for doc in sim_docs:
    print(doc)

page_content='. First, existing STGNNs and spatio-temporal modeling methods [17], [18], [19], [20], [21] typically model spatial and temporal dependencies inde- pendently or alternatively, as illustrated in Fig. 1(C1l) and (C2). In the independent mode, distinct temporal and spa- tial modules are utilized in parallel to capture spatial and temporal dependencies, followed by their combination to comprehend the overall spatio-temporal dependencies within' metadata={'languages': '["eng"]', 'detection_class_prob': 0.951858639717102, 'source': 'knowledge_base\\Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf', 'last_modified': '2026-03-17T15:16:41', 'category': 'NarrativeText', 'element_id': 'bdcb80e856eaf4f0fbe07527a0622606', 'filetype': 'application/pdf', 'page_number': 2, 'coordinates': '{"points": [[137.01365661621094, 1654.863037109375], [137.01365661621094, 2077.257568359375], [834.2998657226562, 2077.257568359375], [834.2998657226562, 1654.863037109

You can also use Maximal Marginal Relevance (MMR) retrieval.
MMR keeps relevance while reducing redundancy among returned documents, improving diversity in retrieved context.

In [14]:
# Maximal Marginal Relevance (MMR) search
query = "How does the SpatialTemporal Co-Attention method model temporal and spatial information?"
sim_docs = vectorstore.max_marginal_relevance_search(query, k=3)
for doc in sim_docs:
    print(doc)

page_content='. First, existing STGNNs and spatio-temporal modeling methods [17], [18], [19], [20], [21] typically model spatial and temporal dependencies inde- pendently or alternatively, as illustrated in Fig. 1(C1l) and (C2). In the independent mode, distinct temporal and spa- tial modules are utilized in parallel to capture spatial and temporal dependencies, followed by their combination to comprehend the overall spatio-temporal dependencies within' metadata={'file_directory': 'knowledge_base', 'filename': 'Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf', 'languages': '["eng"]', 'detection_class_prob': 0.951858639717102, 'element_id': 'bdcb80e856eaf4f0fbe07527a0622606', 'last_modified': '2026-03-17T15:16:41', 'coordinates': '{"points": [[137.01365661621094, 1654.863037109375], [137.01365661621094, 2077.257568359375], [834.2998657226562, 2077.257568359375], [834.2998657226562, 1654.863037109375]], "system": "PixelSpace", "layout_width": 1700, "la

You can also build a retriever from the vector store and query through that retriever.

In [15]:
# Retrieve via a retriever object
query = "How does the SpatialTemporal Co-Attention method model temporal and spatial information?"
# Build retriever from vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",  # options: similarity or mmr
    search_kwargs={"k": 3},
)
sim_docs = retriever.invoke(query)

for doc in sim_docs:
    print(doc)

page_content='. First, existing STGNNs and spatio-temporal modeling methods [17], [18], [19], [20], [21] typically model spatial and temporal dependencies inde- pendently or alternatively, as illustrated in Fig. 1(C1l) and (C2). In the independent mode, distinct temporal and spa- tial modules are utilized in parallel to capture spatial and temporal dependencies, followed by their combination to comprehend the overall spatio-temporal dependencies within' metadata={'file_directory': 'knowledge_base', 'detection_class_prob': 0.951858639717102, 'page_number': 2, 'last_modified': '2026-03-17T15:16:41', 'element_id': 'bdcb80e856eaf4f0fbe07527a0622606', 'source': 'knowledge_base\\Dynamic_Graph_Representation_Learning_for_Spatio-Temporal_Neuroimaging_Analysis.pdf', 'coordinates': '{"points": [[137.01365661621094, 1654.863037109375], [137.01365661621094, 2077.257568359375], [834.2998657226562, 2077.257568359375], [834.2998657226562, 1654.863037109375]], "system": "PixelSpace", "layout_width": 1

## 2.4 Generation
### 2.4.1 Build the Retrieval-to-Generation Chain
After retrieving relevant chunks for a user query, we build a prompt with context + query and send it to the LLM.

We can combine these steps into a single LCEL chain.
LCEL (LangChain Expression Language) is a concise declarative syntax for composing components with a pipeline-style operator.
It also supports streaming, batching, and logging patterns.

Each LCEL component implements the Runnable interface (`invoke`, `batch`, `stream`, `ainvoke`, etc.), so chained objects remain runnable as well.

Create a `.env` file and set your API key:

`TONGYI_API_KEY=your_api_key`

Using `.env` is convenient for local development; in production, use environment variables managed by your deployment platform.

First, load the embedding model and vector store, then build a retriever.

In [16]:
import os
import torch
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_community.llms.tongyi import Tongyi
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.passthrough import RunnablePassthrough

# Load embedding model.
embedding_model = HuggingFaceEmbeddings(
    model_name="./bge-base-zh-v1.5",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Initialize Chroma client.
vectorstore = Chroma(
    persist_directory="vectorstore",
    embedding_function=embedding_model,
)

# Build retriever.
retriever = vectorstore.as_retriever(
    search_type="similarity",  # options: similarity or mmr
    search_kwargs={"k": 3},
)

Build a retrieval-augmented generation flow: retrieve context first, then generate answers with the LLM.

In [17]:
# Retrieval + generation chain
load_dotenv()
TONGYI_API_KEY = os.getenv("TONGYI_API_KEY")

# Join retrieved document contents into one context string.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt template
prompt = PromptTemplate(
    input_variables=["context", "query"],
    template="""
You are a professional academic QA assistant.
Answer the question only based on the provided context.
If the answer is not available from the context, reply: "I don't know".

Context: {context}

Question: {query}

Answer:""",
)

# LLM
llm = Tongyi(model="qwen-turbo", api_key=TONGYI_API_KEY)

# RAG chain
rag_chain = (
    {"context": retriever | format_docs, "query": RunnablePassthrough()}
    | prompt
    | (lambda x: print(x.text, end="") or x)
    | llm
    | StrOutputParser()  # Parse model output as string
)

query = "How does the SpatialTemporal Co-Attention method model temporal and spatial information?"
response = rag_chain.invoke(query)
print(response)


You are a professional academic QA assistant.
Answer the question only based on the provided context.
If the answer is not available from the context, reply: "I don't know".

Context: . First, existing STGNNs and spatio-temporal modeling methods [17], [18], [19], [20], [21] typically model spatial and temporal dependencies inde- pendently or alternatively, as illustrated in Fig. 1(C1l) and (C2). In the independent mode, distinct temporal and spa- tial modules are utilized in parallel to capture spatial and temporal dependencies, followed by their combination to comprehend the overall spatio-temporal dependencies within

GA < SA module in Fig. 7(b) uses the temporal feature attention score to guide the generation of spatial feature representation through the attention transition unit. As shown in Fig. 7(c), the GA GA module fully takes advantage of the learned attention score from each input stream to generate the feature representation mutually. In this scenario, the GA % GA module pr

### 2.4.2 Add Conversation History
In the previous pipeline, the LLM does not persist memory by itself.
Without explicit history, responses may lose conversational continuity.

We can append each turn to a history list and pass recent history into the prompt.

Because context length is limited, keep only the most recent turns instead of sending the full history every time.

In [18]:
# History-aware RAG
load_dotenv()
TONGYI_API_KEY = os.getenv("TONGYI_API_KEY")

# Join retrieved document contents into one context string.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt template
prompt = PromptTemplate(
    input_variables=["context", "history", "query"],
    template="""
You are a professional academic QA assistant.
Answer the question only based on the provided context and conversation history.
If the answer is not available, reply: "I don't know".

Context: {context}

History: [{history}]

Question: {query}

Answer:""",
)

# LLM
llm = Tongyi(model="qwen-turbo", api_key=TONGYI_API_KEY)

# Conversation history
history = []

# Format recent history
def format_history(history):
    # Keep only the latest 3 turns
    max_epoch = 3
    if len(history) > 2 * max_epoch:
        history = history[-2 * max_epoch :]
    return "\n".join([f"{i['role']}: {i['content']}" for i in history])

# RAG chain
rag_chain = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["query"], k=3)),
        "history": lambda x: format_history(x["history"]),
        "query": lambda x: x["query"],
    }
    | prompt
    | (lambda x: print(x.text, end="") or x)
    | llm
    | StrOutputParser()  # Parse model output as string
)

query_list = [
    "How is the SpatialTemporal Co-Attention mechanism constructed?",
    "Which paper uses federated learning, and what benefit does it claim?",
    "How does the SpatialTemporal Co-Attention method model temporal and spatial information?",
    "If we have small-sample MRI data, which method is more suitable and why?",
]
for query in query_list:
    print(f"===== Query: {query} =====")
    response = rag_chain.invoke({"query": query, "history": history})
    print(response, end="\n\n")
    history.extend(
        [
            {"role": "user", "content": query},
            {"role": "assistant", "content": response},
        ]
    )

===== Query: How does the SpatialTemporal Co-Attention method model temporal and spatial information? =====

You are a professional academic QA assistant.
Answer the question only based on the provided context and conversation history.
If the answer is not available, reply: "I don't know".

Context: . First, existing STGNNs and spatio-temporal modeling methods [17], [18], [19], [20], [21] typically model spatial and temporal dependencies inde- pendently or alternatively, as illustrated in Fig. 1(C1l) and (C2). In the independent mode, distinct temporal and spa- tial modules are utilized in parallel to capture spatial and temporal dependencies, followed by their combination to comprehend the overall spatio-temporal dependencies within

GA < SA module in Fig. 7(b) uses the temporal feature attention score to guide the generation of spatial feature representation through the attention transition unit. As shown in Fig. 7(c), the GA GA module fully takes advantage of the learned attention s

### 2.4.3 Query Rewriting
Even with history-aware prompts, users often ask follow-up questions using pronouns or omitted entities.
Directly retrieving with such short queries may miss the correct context.

To improve retrieval quality, we first rewrite the user query using conversation history (coreference resolution / entity completion), then retrieve with the rewritten query.

If no history exists, use the original query directly.

In [19]:
# Query rewriting before retrieval
load_dotenv()
TONGYI_API_KEY = os.getenv("TONGYI_API_KEY")

# Join retrieved document contents into one context string.
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Prompt template for final answering
prompt = PromptTemplate(
    input_variables=["context", "history", "query"],
    template="""
You are a professional academic QA assistant.
Answer only based on the provided context and history.
If the answer is not available, reply: "I don't know".

Context: {context}

History: [{history}]

Question: {query}

Answer:""",
)

# LLM
llm = Tongyi(model="qwen-turbo", api_key=TONGYI_API_KEY)

# Conversation history
history = []

# Format recent history
def format_history(history):
    # Keep only the latest 3 turns
    max_epoch = 3
    if len(history) > 2 * max_epoch:
        history = history[-2 * max_epoch :]
    return "\n".join([f"{i['role']}: {i['content']}" for i in history])

# Rephrase prompt template
rephrase_prompt = PromptTemplate(
    input_variables=["history", "query"],
    template="""
Rewrite the user query using conversation history to make it more specific.
Output only the rewritten query.

History: [{history}]

Query: {query}
""",
)

# Rephrase chain: generate a clearer query from history + current query
rephrase_chain = (
    {
        "history": lambda x: format_history(x["history"]),
        "query": lambda x: x["query"],
    }
    | rephrase_prompt
    | llm
    | StrOutputParser()
    | (lambda x: print(f"===== Rewritten Query: {x} =====") or x)
)

# RAG chain
rag_chain = (
    {
        "context": lambda x: format_docs(
            retriever.invoke(
                rephrase_chain.invoke({"history": x["history"], "query": x["query"]})
                if x["history"]
                else x["query"],
                k=3,
            )
        ),
        "history": lambda x: format_history(x["history"]),
        "query": lambda x: x["query"],
    }
    | prompt
    | (lambda x: print(x.text, end="") or x)
    | llm
    | StrOutputParser()  # Parse model output as string
)

query_list = [
    "How is the SpatialTemporal Co-Attention mechanism constructed?",
    "Which paper uses federated learning, and what benefit does it claim?",
    "How does the SpatialTemporal Co-Attention method model temporal and spatial information?",
    "If we have small-sample MRI data, which method is more suitable and why?",
]
for query in query_list:
    print(f"===== Query: {query} =====")
    response = rag_chain.invoke({"query": query, "history": history})
    print(response, end="\n\n")
    history.extend(
        [
            {"role": "user", "content": query},
            {"role": "assistant", "content": response},
        ]
    )

===== Query: How does the SpatialTemporal Co-Attention method model temporal and spatial information? =====

You are a professional academic QA assistant.
Answer only based on the provided context and history.
If the answer is not available, reply: "I don't know".

Context: . First, existing STGNNs and spatio-temporal modeling methods [17], [18], [19], [20], [21] typically model spatial and temporal dependencies inde- pendently or alternatively, as illustrated in Fig. 1(C1l) and (C2). In the independent mode, distinct temporal and spa- tial modules are utilized in parallel to capture spatial and temporal dependencies, followed by their combination to comprehend the overall spatio-temporal dependencies within

GA < SA module in Fig. 7(b) uses the temporal feature attention score to guide the generation of spatial feature representation through the attention transition unit. As shown in Fig. 7(c), the GA GA module fully takes advantage of the learned attention score from each input strea